 # Hyperparameter Tuning for CatBoost and LightGBM

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform
from joblib import dump
import warnings
warnings.filterwarnings("ignore")

SEED = 42

processed_data_dir = os.path.join('..', 'data', 'processed')
output_dir = os.path.join('..', 'models', 'tuned_models')
os.makedirs(output_dir, exist_ok=True)

best_file_name = "Data_iterative_imputed.pkl"
file_path = os.path.join(processed_data_dir, best_file_name)
Data = pd.read_pickle(file_path)

X = Data.drop(columns=['target'])
y = Data['target']

print(f"✅ Dataset loaded: {best_file_name} | Shape: {X.shape}")

catboost_params = {
    'iterations': randint(100, 500),
    'learning_rate': uniform(0.01, 0.3),
    'depth': randint(4, 10),
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 255),
    'random_strength': uniform(1e-5, 10),
    'bagging_temperature': uniform(0.1, 1.0),
}

lightgbm_params = {
    'num_leaves': randint(20, 100),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.2),
    'n_estimators': randint(50, 300),
    'subsample': uniform(0.6, 1.0),
    'colsample_bytree': uniform(0.6, 1.0),
    'reg_alpha': uniform(0.0, 1.0),
    'reg_lambda': uniform(0.0, 1.0),
    'class_weight': [None, 'balanced']
}

def run_tuning(X, y, model, param_dist, model_name, cv=3, n_iter=20):
    print(f"\n🚀 Running hyperparameter tuning for {model_name}")
    
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring='roc_auc',
        cv=skf,
        n_jobs=-1,
        verbose=0,
        random_state=SEED
    )
    
    search.fit(X, y)
    
    best_params = search.best_params_
    best_score = search.best_score_
    
    result_summary = {
        'Model': model_name,
        'Best Score (ROC AUC)': best_score,
        'Best Parameters': best_params
    }
    
    print(f"\n🏆 Best ROC AUC ({model_name}): {best_score:.4f}")
    print("🧾 Best parameters:")
    print(best_params)
    
    return result_summary, search.best_estimator_

imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_imp = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imp)

catboost_model = CatBoostClassifier(auto_class_weights='Balanced', verbose=False, random_state=SEED)
catboost_summary, best_catboost = run_tuning(
    X_scaled, y,
    catboost_model,
    catboost_params,
    "CatBoost",
    cv=3,
    n_iter=20
)

lightgbm_model = LGBMClassifier(random_state=SEED)
lightgbm_summary, best_lightgbm = run_tuning(
    X_scaled, y,
    lightgbm_model,
    lightgbm_params,
    "LightGBM",
    cv=3,
    n_iter=20
)

tuning_summary = pd.DataFrame([catboost_summary, lightgbm_summary])
print("\n📊 Summary of best models after tuning:")
print(tuning_summary[['Model', 'Best Score (ROC AUC)', 'Best Parameters']])

dump(imputer, os.path.join(output_dir, f'imputer_{best_file_name}.joblib'))
dump(scaler, os.path.join(output_dir, f'scaler_{best_file_name}.joblib'))

dump(best_catboost, os.path.join(output_dir, f'CatBoost_tuned.joblib'))
dump(best_lightgbm, os.path.join(output_dir, f'LightGBM_tuned.joblib'))

print(f"\n💾 Best models and preprocessors saved to: {output_dir}")

✅ Dataset loaded: Data_iterative_imputed.pkl | Shape: (31188, 80)

🚀 Running hyperparameter tuning for CatBoost

🏆 Best ROC AUC (CatBoost): 0.7763
🧾 Best parameters:
{'bagging_temperature': 0.6208342600258236, 'border_count': 221, 'depth': 8, 'iterations': 307, 'l2_leaf_reg': 8.473201101373808, 'learning_rate': 0.17190763971672393, 'random_strength': 5.867521656638482}

🚀 Running hyperparameter tuning for LightGBM
[LightGBM] [Info] Number of positive: 902, number of negative: 30286
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020069 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11679
[LightGBM] [Info] Number of data points in the train set: 31188, number of used features: 79
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.028921 -> initscore=-3.513826
[LightGBM] [Info] Start training from score -3.513826
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

In [7]:
pip install streamlit

   ---------------------------------------- 0.0/413.4 kB ? eta -:--:--
    --------------------------------------- 10.2/413.4 kB ? eta -:--:--
   ----- --------------------------------- 61.4/413.4 kB 656.4 kB/s eta 0:00:01
   ------------------ --------------------- 194.6/413.4 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------  409.6/413.4 kB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 413.4/413.4 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/63.8 kB ? eta -:--:--
   ---------------------------------------- 63.8/63.8 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.30.2
    Uninstalling protobuf-6.30.2:
      Successfully uninstalled protobuf-6.30.2
  Attempting uninstall: attrs
    Found existing installation: attrs 18.2.0
    Uninstalling attrs-18.2.0:
      Successfully uninstalled attrs-18.2.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdflatex 0.1.3 requires attrs<19.0,>=18.2, but you have attrs 25.3.0 which is incompatible.
